# Продвинутое ООП и паттерны проектирования


На прошлой лекции мы научились описывать предметную область с помощью классов. Теперь посмотрим, как строить систему из нескольких взаимодействующих классов так, чтобы её было удобно расширять.

## Разнородные объекты и проблема условий


Начнём с двух независимых классов. Оба представляют расходы, но хранят разные данные и по-разному вычисляют сумму.

In [ ]:
from dataclasses import dataclass

@dataclass
class Purchase:
    title: str
    price: float
    quantity: int

    def calculate(self) -> float:
        return self.price * self.quantity


@dataclass
class Subscription:
    title: str
    monthly_price: float
    months: int

    def calculate(self) -> float:
        return self.monthly_price * self.months

И применение:

In [ ]:
coffee = Purchase("Кофе", price=8.5, quantity=2)
music = Subscription("Музыка", monthly_price=9.99, months=3)

print(coffee.calculate())
print(music.calculate())

17.0
29.97


Представим функцию, которая сама проверяет тип каждого объекта и повторяет формулы классов.

In [ ]:
def calculate_total_bad(items: list) -> float:
    total = 0.0

    for item in items:
        if isinstance(item, Purchase):
            # item.calculate_purchase()
            total += item.price * item.quantity
        elif isinstance(item, Subscription):
            # item.calculate_sub()
            total += item.monthly_price * item.months

    return total


calculate_total_bad([coffee, music])

46.97

Такое решение работает, но плохо расширяется:

- функция знает внутреннее устройство каждого класса;
- при добавлении нового типа расхода функцию придётся менять;
- цепочка `if/elif` будет расти;
- одна и та же формула окажется записана в нескольких местах.

## Полиморфизм и duck typing


У объектов уже есть общий способ получить сумму — метод `.calculate()`. Функции достаточно пользоваться этим методом и не проверять конкретный класс.

In [ ]:
def calculate_total(items: list) -> float:
    return sum(item.calculate() for item in items)


calculate_total([coffee, music])

46.97

**Полиморфизм** позволяет одинаково обращаться к разным объектам через общий интерфейс.

Python часто использует **duck typing**: важен не конкретный класс объекта, а доступное поведение. Если объект предоставляет подходящий метод `.calculate()`, функция сможет с ним работать.

In [ ]:
@dataclass
class Service:
    title: str
    hourly_rate: float
    hours: float

    def calculate(self) -> float:
        return self.hourly_rate * self.hours


repair = Service("Ремонт", hourly_rate=30, hours=2)
calculate_total([coffee, music, repair])

106.97

Функцию `calculate_total()` изменять не пришлось. Новый класс поддерживает ожидаемое поведение и поэтому участвует в вычислении.

Попробуем создать класс `Fine`, который также сработает в `calculate_total()`:

In [ ]:
@dataclass
class Fine:
    title: str
    amount: float

    def calculate(self) -> float:
        return self.amount

fine = Fine("ГАИ", 10.5)

calculate_total([coffee, music, repair, fine])

117.47

## Абстрактные классы


Duck typing удобен, но не гарантирует наличие нужного метода заранее. Абстрактный класс позволяет явно зафиксировать обязательный интерфейс семейства классов.

In [ ]:
from abc import ABC, abstractmethod

class Expense(ABC):
    @abstractmethod
    def calculate(self) -> float:
        raise NotImplementedError

`ABC` означает *Abstract Base Class*. Декоратор `@abstractmethod` отмечает метод, который должны реализовать дочерние классы.

Создать объект незавершённого абстрактного класса нельзя. Проверим это, не оставляя в ноутбуке необработанную ошибку.

In [ ]:
try:
    Expense()
except TypeError as error:
    print(type(error).__name__)
    print(error)

TypeError
Can't instantiate abstract class Expense without an implementation for abstract method 'calculate'


Свяжем наши классы с абстрактным классом `Expense`:

In [ ]:
@dataclass
class Purchase(Expense):
    title: str
    price: float
    quantity: int = 1

    def calculate(self) -> float:
        return self.price * self.quantity


@dataclass
class Subscription(Expense):
    title: str
    monthly_price: float
    months: int = 1

    def calculate(self) -> float:
        return self.monthly_price * self.months


@dataclass
class Service(Expense):
    title: str
    hourly_rate: float
    hours: float

    def calculate(self) -> float:
        return self.hourly_rate * self.hours

In [ ]:
@dataclass
class Charge(Expense):
    value: float


Charge()

TypeError: Can't instantiate abstract class Charge without an implementation for abstract method 'calculate'

И применим:

In [ ]:
expenses: list[Expense] = [
    Purchase("Книга", 35),
    Subscription("Музыка", 9.99, 3),
    Service("Ремонт", 30, 2),
]

calculate_total(expenses)

Аннотация `list[Expense]` показывает намерение: коллекция содержит разные конкретные объекты, объединённые общим контрактом `Expense`.

Абстрактный класс не отменяет duck typing, а делает архитектурное требование явным.

## Паттерн Template Method


Теперь у всех расходов есть один метод, но его реализация полностью дублирует общий процесс. Пусть порядок расчёта всегда состоит из четырёх шагов:

1. Рассчитать базовую сумму.
2. Применить дополнительные правила.
3. Проверить результат.
4. Вернуть сумму.

**Template Method** фиксирует общий алгоритм в базовом классе, оставляя отдельные шаги дочерним классам.

In [ ]:
class Expense(ABC):
    def calculate(self) -> float:
        amount = self.calculate_base_amount()
        amount = self.apply_adjustments(amount)
        self.validate_amount(amount)
        return amount

    @abstractmethod
    def calculate_base_amount(self) -> float:
        raise NotImplementedError

    def apply_adjustments(self, amount: float) -> float:
        return amount

    def validate_amount(self, amount: float) -> None:
        if amount < 0:
            raise ValueError("Сумма не может быть отрицательной")

`.calculate()` — шаблонный метод: он определяет последовательность действий. `.calculate_base_amount()` — обязательный изменяемый шаг. `.apply_adjustments()` — необязательная точка расширения с готовой реализацией.

In [ ]:
@dataclass
class Purchase(Expense):
    title: str
    price: float
    quantity: int = 1

    def calculate_base_amount(self) -> float:
        return self.price * self.quantity

    # def validate_amount(self, amount: float) -> None:
    #     if self.title == "Книга" and amount > 30:
    #         raise ValueError("Книга должна быть дешевле 30")

@dataclass
class Subscription(Expense):
    title: str
    monthly_price: float
    months: int = 1

    def calculate_base_amount(self) -> float:
        return self.monthly_price * self.months


@dataclass
class Service(Expense):
    title: str
    hourly_rate: float
    hours: float

    def calculate_base_amount(self) -> float:
        return self.hourly_rate * self.hours

И применение:

In [ ]:
book = Purchase("Книга", price=35)
repair = Service("Ремонт", hourly_rate=30, hours=2)

print(repair.calculate())
print(book.calculate())


60


ValueError: Книга должна быть дешевле 30

In [ ]:
calculate_total([book, repair])

95

Валидация теперь выполняется централизованно для всех дочерних классов.

In [ ]:
try:
    Purchase("Ошибка в данных", price=-10).calculate()
except ValueError as error:
    print(error)

## Наследование или композиция


Наследование выражает отношение **"является разновидностью"**, а композиция — **"содержит"** или **"использует"**.

```text
Purchase является Expense          → наследование
Report содержит Expense            → композиция
Purchase использует Discount       → композиция
```

Если создавать отдельный дочерний класс для каждого вида скидки, иерархия быстро разрастается:

```text
Purchase
├── PurchaseWithPercentDiscount
├── PurchaseWithFixedDiscount
├── PurchaseWithBonusDiscount
└── PurchaseWithoutDiscount
```

Скидка — не разновидность покупки, а заменяемый алгоритм. Значит, здесь лучше композиция.

## Паттерн Strategy


**Strategy** выносит взаимозаменяемые алгоритмы в отдельные классы с общим интерфейсом. Контекст использует выбранную стратегию, не зная деталей её реализации.

В нашем примере контекст — `Purchase`, а стратегии — способы применения скидки.

In [ ]:
class DiscountStrategy(ABC):
    @abstractmethod
    def apply(self, amount: float) -> float:
        pass

Его разновидности:

In [ ]:
@dataclass
class NoDiscount(DiscountStrategy):
    def apply(self, amount: float) -> float:
        return amount


@dataclass
class PercentDiscount(DiscountStrategy):
    percent: float

    def apply(self, amount: float) -> float:
        return amount * (1 - self.percent / 100)


@dataclass
class FixedDiscount(DiscountStrategy):
    value: float

    def apply(self, amount: float) -> float:
        return max(0, amount - self.value)

И применение:

In [ ]:
from dataclasses import field

@dataclass
class Purchase(Expense):
    title: str
    price: float
    quantity: int = 1
    discount_type: str = "no_discount"
    discount: DiscountStrategy = field(default_factory=NoDiscount)

    def calculate_base_amount(self) -> float:
        return self.price * self.quantity

    def apply_adjustments(self, amount: float) -> float:
        return self.discount.apply(amount)

In [ ]:
sneakers = Purchase("Кроссовки", price=200)
print("Без скидки:", sneakers.calculate())

sneakers.discount = PercentDiscount(10)
print("Скидка 10%:", sneakers.calculate())

sneakers.discount = FixedDiscount(30)
print("Скидка 30 BYN:", sneakers.calculate())

Без скидки: 200
Скидка 10%: 180.0
Скидка 30 BYN: 170


Без стратегии:

In [ ]:
@dataclass
class Purchase(Expense):
    title: str
    price: float
    quantity: int = 1

    def calculate_base_amount(self) -> float:
        return self.price * self.quantity

    def apply_adjustments(self, amount: float) -> float:
        if ...:
            return amount
        elif ...:
            return max(0, amount - ...)
        elif ...:
            return (amount * (1 - .../100))
        elif ...:
            ...

Объект покупки не изменился, но используемый алгоритм был заменён во время выполнения. При добавлении новой стратегии класс `Purchase` менять не потребуется.


Давайте реализуем `LimitDiscount`. Она принимает другую стратегию и ограничивает размер получившейся скидки значением `max_discount`.

In [ ]:
@dataclass
class LimitDiscount(DiscountStrategy):
    strategy: DiscountStrategy
    max_discount: float

    def apply(self, amount: float) -> float:
        new_amount = self.strategy.apply(amount)
        discount = amount - new_amount
        if discount > self.max_discount:
            return amount - self.max_discount
        else:
            return new_amount

In [ ]:
book = Purchase(
    title="Гарри Поттер",
    price=10.0,
    quantity=5,
    discount=LimitDiscount(
        strategy=PercentDiscount(20),
        max_discount=20
    )
)

book.calculate()

40.0

## Обычные, классовые и статические методы


| Вид метода | Первый параметр | Основное назначение |
|---|---|---|
| Обычный | `self` | Работа с конкретным объектом |
| Классовый | `cls` | Альтернативное создание объекта или работа с классом |
| Статический | отсутствует | Вспомогательная операция, связанная с классом |

Обычный метод `.calculate_base_amount()` использует состояние объекта. Статический `.validate_amount()` не зависит ни от объекта, ни от конкретного класса. Классовый метод удобно использовать как альтернативный конструктор.

In [ ]:
@dataclass
class Purchase(Expense):
    title: str
    price: float
    quantity: int = 1
    discount: DiscountStrategy = field(default_factory=NoDiscount)

    def calculate_base_amount(self) -> float:
        return self.price * self.quantity

    def apply_adjustments(self, amount: float) -> float:
        return self.discount.apply(amount)

    @classmethod
    def from_dict(cls, data: dict) -> Purchase:  # data={"type": "purchase", "title": "Кофе", "price": 8.5, "quantity": 2}
        return cls(   # cls = Purchase
            title=data["title"],
            price=float(data["price"]),
            quantity=int(data.get("quantity", 1)),
        )
        # return Purchase(title=data["title"], price=float(data["price"]), quantity=int(data.get("quantity", 1)))

    @staticmethod
    def smth():
        return 10

def smth():
    return 10

In [ ]:
data = {
    "type": "purchase",
    "title": "Кофе",
    "price": 8.5,
    "quantity": 2,
}

coffee = Purchase.from_dict(data)
coffee

Purchase(title='Кофе', price=8.5, quantity=2, discount=NoDiscount())

In [ ]:
Purchase.smth()
coffee.smth()

10

`.from_dict()` получает класс в параметре `cls`, поэтому при наследовании создаст объект того класса, через который был вызван.

Не каждую вспомогательную функцию стоит превращать в `@staticmethod`. Если операция не связана с ответственностью класса, лучше оставить её функцией модуля.

## Паттерн Factory



После чтения JSON мы получаем словари. По значению поля `type` нужно выбрать подходящий класс. Клиентскому коду не обязательно знать правила создания каждого объекта.

In [ ]:
@dataclass
class Subscription(Expense):
    title: str
    monthly_price: float
    months: int = 1

    def calculate_base_amount(self) -> float:
        return self.monthly_price * self.months

    @classmethod
    def from_dict(cls, data: dict) -> Subscription:
        return cls(
            title=data["title"],
            monthly_price=float(data["monthly_price"]),
            months=int(data.get("months", 1)),
        )


@dataclass
class Service(Expense):
    title: str
    hourly_rate: float
    hours: float

    def calculate_base_amount(self) -> float:
        return self.hourly_rate * self.hours

    @classmethod
    def from_dict(cls, data: dict) -> 'Service':
        return cls(
            title=data["title"],
            hourly_rate=float(data["hourly_rate"]),
            hours=float(data.get("hours", 1)),
        )

    @classmethod
    def plumber(cls, hours: float) -> 'Service':
        return cls(
            title="Plumbing",
            hourly_rate=10.0,
            hours=hours,
        )

In [ ]:
Service.plumber(5)


Service(title='Plumbing', hourly_rate=10.0, hours=5)

In [ ]:
plumbing = Service(title='Plumbing', hourly_rate=10.0, hours=5)
plumbing.hourly_rate * plumbing.hours

Создадим на лету нужный класс:

In [ ]:
EXPENSE_TYPES: dict[str, type[Expense]] = {
    "purchase": Purchase,
    "subscription": Subscription,
    "service": Service,
}


def create_expense(data: dict) -> Expense:  # data={"type": "purchase", "title": "Кофе", "price": 8.5, "quantity": 2}
    expense_class = EXPENSE_TYPES.get(data["type"])  # expense_class = Purchase

    if expense_class is None:
        raise ValueError(f"Неизвестный вид расхода: {data['type']}")
    instance = expense_class.from_dict(data)  # Purchase.from_dict({"type": "purchase", "title": "Кофе", "price": 8.5, "quantity": 2})
    return instance  # Purchase(title=data["title"], price=float(data["price"]), quantity=int(data.get("quantity", 1)))

Проверим:

In [ ]:
import json

json_text = """[
  {"type": "purchase", "title": "Кофе", "price": 8.5, "quantity": 2},
  {"type": "subscription", "title": "Музыка", "monthly_price": 9.99, "months": 3},
  {"type": "service", "title": "Ремонт", "hourly_rate": 30, "hours": 2}
]"""

raw_expenses = json.loads(json_text)
expenses = [create_expense(item) for item in raw_expenses]

print(expenses)

[Purchase(title='Кофе', price=8.5, quantity=2, discount=NoDiscount()), Subscription(title='Музыка', monthly_price=9.99, months=3), Service(title='Ремонт', hourly_rate=30.0, hours=2.0)]


Здесь использована **простая фабрика**: функция выбирает класс и возвращает объект общего типа `Expense`.

- `.from_dict()` — альтернативный конструктор;
- `.create_expense()` — простая фабрика;
- классический Factory Method использует полиморфный метод создания в иерархии создателей.

Для текущей задачи простой фабрики достаточно.

In [ ]:
try:
    create_expense({"type": "unknown", "title": "?"})
except ValueError as error:
    print(error)

Продемонстрируем силу полиморфизма.

Можем создать класс `Report` содержит объекты общего типа `Expense` и вызывает их публичный метод. Он не знает формул покупки, подписки или услуги.

In [ ]:
@dataclass
class Report:
    title: str
    expenses: list[Expense] = field(default_factory=list)

    def add(self, expense: Expense) -> None:
        self.expenses.append(expense)

    @property
    def total(self) -> float:
        return sum(expense.calculate() for expense in self.expenses)

In [ ]:
report = Report("Октябрь", expenses)

print(report.expenses)
print(f"Итого: {report.total:.2f} BYN")

[Purchase(title='Кофе', price=8.5, quantity=2, discount=NoDiscount()), Subscription(title='Музыка', monthly_price=9.99, months=3), Service(title='Ремонт', hourly_rate=30.0, hours=2.0)]
Итого: 106.97 BYN


Ответственность разделена между объектами:

```text
JSON
  ↓
Factory: выбор и создание объекта
  ↓
Purchase / Subscription / Service
  ↓
Template Method: общий порядок расчёта
  ↓
Strategy: заменяемый способ скидки
  ↓
Report: полиморфная обработка
```

## Специальные методы как протоколы Python


Специальные методы позволяют нашим объектам участвовать во встроенных операциях:

- `__len__` позволяет вызвать `len()`;
- `__lt__` используется сравнением `<` и сортировкой;
- `__add__` определяет поведение оператора `+`.

Соберём финальную версию модели. Сначала расширим базовый класс строковым представлением и сравнением.

In [ ]:
class Expense(ABC):
    title: str

    def calculate(self) -> float:
        amount = self.calculate_base_amount()
        amount = self.apply_adjustments(amount)
        self.validate_amount(amount)
        return amount

    @abstractmethod
    def calculate_base_amount(self) -> float:
        """Ожидаем подсчет базовой стоимости траты"""

    def apply_adjustments(self, amount: float) -> float:
        return amount

    @staticmethod
    def validate_amount(amount: float) -> None:
        if amount < 0:
            raise ValueError("Сумма не может быть отрицательной")

    def __lt__(self, other: object) -> bool:
        if not isinstance(other, Expense):
            return NotImplemented
        return self.calculate() < other.calculate()


    def __eq__(self, other: object) -> bool:
        if not isinstance(other, Expense):
            return NotImplemented
        return self.calculate() == other.calculate()

In [ ]:
@dataclass(eq=False, unsafe_hash=True)
class Purchase(Expense):
    title: str
    price: float
    quantity: int = 1
    discount: DiscountStrategy = field(default_factory=NoDiscount, hash=False)

    def calculate_base_amount(self) -> float:
        return self.price * self.quantity

    def apply_adjustments(self, amount: float) -> float:
        return self.discount.apply(amount)

    @classmethod
    def from_dict(cls, data: dict) -> Purchase:
        return cls(
            title=data["title"],
            price=float(data["price"]),
            quantity=int(data.get("quantity", 1)),
        )


@dataclass(eq=False, unsafe_hash=True)
class Subscription(Expense):
    title: str
    monthly_price: float
    months: int = 1

    def calculate_base_amount(self) -> float:
        return self.monthly_price * self.months

    @classmethod
    def from_dict(cls, data: dict) -> Subscription:
        return cls(
            title=data["title"],
            monthly_price=float(data["monthly_price"]),
            months=int(data.get("months", 1)),
        )


@dataclass(eq=False, unsafe_hash=True)
class Service(Expense):
    title: str
    hourly_rate: float
    hours: float

    def calculate_base_amount(self) -> float:
        return self.hourly_rate * self.hours

    @classmethod
    def from_dict(cls, data: dict) -> Service:
        return cls(
            title=data["title"],
            hourly_rate=float(data["hourly_rate"]),
            hours=float(data["hours"]),
        )

In [ ]:
book = Purchase(title="Book", price=10, quantity=3)
netflix = Subscription(title="Netfix", monthly_price=10, months=3)
plumber = Service(title="Plumber", hourly_rate=10, hours=3)

plumber > netflix

False

In [ ]:
@dataclass
class Report:
    title: str
    expenses: list[Expense] = field(default_factory=list)

    def add(self, expense: Expense) -> None:
        self.expenses.append(expense)

    @property
    def total(self) -> float:
        return sum(expense.calculate() for expense in self.expenses)

    def __len__(self) -> int:
        return len(self.expenses)

    def __str__(self) -> str:
        return f"{self.title}: {len(self)} операций, {self.total} BYN"

    def __hash__(self) -> int:
        return hash((self.title, tuple(self.expenses)))

    def __add__(self, other: object) -> Report:
        if not isinstance(other, Report):
            return NotImplemented
        return Report(
            title=f"{self.title} + {other.title}",
            expenses=self.expenses + other.expenses,
        )

In [ ]:
report = Report(
    "Октябрь",
    [create_expense(item) for item in raw_expenses],
)

print(report)
print("Количество:", len(report))
print("По возрастанию суммы:")

for expense in sorted(report.expenses):
    print("-", expense)

Октябрь: 3 операций, 106.97 BYN
Количество: 3
По возрастанию суммы:
- Purchase(title='Кофе', price=8.5, quantity=2, discount=NoDiscount())
- Subscription(title='Музыка', monthly_price=9.99, months=3)
- Service(title='Ремонт', hourly_rate=30.0, hours=2.0)


In [ ]:
november = Report(
    "Ноябрь",
    [Purchase("Книга", 35), Purchase("Чай", 6.5)],
)

combined = report + november
print(combined)

Октябрь + Ноябрь: 5 операций, 148.47 BYN


In [ ]:
hash(november)

6812566462047964944

In [ ]:
{november, november, november}

{Report(title='Ноябрь', expenses=[Purchase(title='Книга', price=35, quantity=1, discount=NoDiscount()), Purchase(title='Чай', price=6.5, quantity=1, discount=NoDiscount())])}

Если операция не поддерживает переданный тип, специальному методу обычно следует вернуть `NotImplemented`. Тогда Python сможет попробовать отражённую операцию или сформировать корректную ошибку типов.

Перегружать оператор стоит только тогда, когда его смысл понятен пользователю класса.

## Множественное наследование, MRO и миксины


**Миксин** — небольшой класс, добавляющий вспомогательное поведение. Обычно он не представляет самостоятельную сущность и не хранит основное состояние объекта.

In [ ]:
class JsonMixin:
    def to_dict(self) -> dict:
        return self.__dict__.copy()


class PrintableMixin:
    def print_info(self) -> None:
        print(self)

In [ ]:
class StupidMixin:
    def stupid(self):
        return self.calculate() ** 2

s = StupidMixin()
s.stupid()

Можем добавить эти новые поведения к классу `Report`:

In [ ]:
class ExportablePurchase(JsonMixin, PrintableMixin, StupidMixin, Purchase):
    pass


exportable = ExportablePurchase("Ноутбук", price=2500)
exportable.print_info()
print(exportable.to_dict())

ExportablePurchase(title='Ноутбук', price=2500, quantity=1, discount=NoDiscount())
{'title': 'Ноутбук', 'price': 2500, 'quantity': 1, 'discount': NoDiscount()}


In [ ]:
exportable.stupid()
# ExportablePurchase.stupid() ->
# StupidMixin.stupid() -> calculate() ->
# Purchase.calculate() ->
# Expense.calculate() -> calculate_base_amount() ->
# Purchase.calculate_base_amount()

6250000

In [ ]:
from dataclasses import dataclass

@dataclass
class Timed:
    created_at: int
    updated_at: int

@dataclass
class Versioned:
    version: int


@dataclass
class User(Timed, Versioned):
    name: str

При множественном наследовании Python ищет методы согласно **MRO** — Method Resolution Order. Порядок можно увидеть через метод `mro()`.

In [3]:
[cls for cls in ExportablePurchase.mro()]

NameError: name 'ExportablePurchase' is not defined

In [10]:
class B:
  def bar(self):
    print("B")

class A(B):
  def foo(self):
    print("A")

  def baz(self):
    print("A")

class C(B):
  def baz(self):
    print("C")

class D(A, B, C):
  pass


d = D()

d.baz()

TypeError: Cannot create a consistent method resolution order (MRO) for bases B, C

In [6]:
D.mro()

[__main__.D, __main__.A, __main__.B, __main__.C, object]

```
B
|\
| C
| |
A |
\ /
 D
```

`super()` также следует MRO, а не просто обращается к классу, записанному первым в определении. Чем сложнее иерархия, тем труднее предсказать взаимодействие методов, поэтому множественное наследование лучше использовать умеренно.

In [17]:
class A:
    def foo(self):
        return {"a": 1}

class B:
    def foo(self):
        return {"b": 1}


class C(A, B):
    pass

class D(C):
    def foo(self):
       dict_ = self.foo()
       dict_["d"] = 1
       return dict_

d = D()
d.foo()

{'a': 1, 'd': 1}

## Дополнительный паттерн: Adapter


**Adapter** преобразует интерфейс существующего объекта в интерфейс, который ожидает наша программа.

Допустим, класс сторонней банковской библиотеки менять нельзя, а его методы называются иначе.

In [18]:
class BankTransaction:
    def get_value(self) -> float:
        return 25.50

    def get_description(self) -> str:
        return "Оплата картой"

In [ ]:
from dataclasses import dataclass

@dataclass
class BankTransactionAdapter(Expense):
    transaction: BankTransaction

    @property
    def title(self):
        return transaction.get_description()

    def calculate_base_amount(self) -> float:
        return self.transaction.get_value()

In [ ]:
transaction = BankTransaction()
adapted_expense = BankTransactionAdapter(transaction)

report.add(adapted_expense)
print(adapted_expense)
print(report)

Адаптер содержит исходный объект и переводит вызовы с одного интерфейса на другой. `Report` продолжает работать только с абстракцией `Expense`.

## Дополнительный паттерн: Composite


**Composite** позволяет одинаково работать с отдельным объектом и группой объектов. Группа реализует тот же интерфейс `Expense`, что и её элементы.

In [ ]:
@dataclass
class ExpenseGroup(Expense):
    title: str
    expenses: list[Expense] = field(default_factory=list)

    def add(self, expense: Expense) -> None:
        self.expenses.append(expense)

    def calculate_base_amount(self) -> float:
        return sum(item.calculate() for item in self.expenses)

In [ ]:
food = ExpenseGroup(
    "Еда",
    [
        Purchase("Кофе", 8.5),
        Purchase("Обед", 18),
    ],
)

weekend = ExpenseGroup(
    "Выходные",
    [
        food,
        Service("Такси", hourly_rate=12, hours=1),
    ],
)

print(food)
print(weekend)

`ExpenseGroup` может содержать обычные расходы и другие группы. Благодаря общему интерфейсу клиентский код не различает лист и составной объект дерева.

## Типичные ошибки


**Проверка типа вместо полиморфизма.** Если код постоянно спрашивает `isinstance()`, возможно, поведение следует перенести в общий метод объектов.

**Слишком глубокое наследование.** Если классы различаются только алгоритмом, попробуйте Strategy и композицию.

**Паттерн без проблемы.** Паттерны добавляют классы и связи. Используйте их, когда они действительно упрощают изменение программы.

**Изменяемое значение по умолчанию.** Для списков и объектов-стратегий в `dataclass` используйте `field(default_factory=...)`.

**Неясная перегрузка операторов.** Не определяйте `+` или `<`, если смысл операции нельзя легко объяснить пользователю класса.

In [ ]:
class A:
    pass

lst = []

def foo():
    a = A()
    lst.append(a)
    del a


foo()
print(lst)


def flask():
    while True:
        try:
            request = input("Жду запроса:")
            if request == "foo":
                print(foo())
        except:
            pass

flask()

Жду запроса:foo
None
Жду запроса:foo
None
Жду запроса:foo
None
Жду запроса:


In [ ]:
from abc import ABC, abstractmethod


class AbstractStrategy(ABC):
    @abstractmethod
    def apply(self):
        pass

class SpecificStrategyA(AbstractStrategy):
    def apply(self):
        return 10


class SpecificStrategyB(AbstractStrategy):
    def apply(self):
        return 20


@dataclass
class Item:
    title: str
    price: int


@dataclass
class Expense:
    item: Item
    strategy: AbstractStrategy


@dataclass
class SuperExpense(Expense):
    pass